# Potato Blight Classification with an Attention CNN

Implementation snapshot accompanying the paper on potato and tomato blight classification. This supplied notebook covers **potato early blight versus late blight only**. The tomato experiment and saved model weights were not provided.

Read `../docs/CODE_REVIEW.md` and `../docs/RUNNING.md` before executing. Original code cells are preserved; saved outputs were removed from this notebook and selected historical artifacts were exported separately with provenance. No training was rerun.


## Colab setup

This cell mounts Google Drive. Local Jupyter users should skip it and update every Drive path in later cells.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Potato model, training, validation, and curves

Trains a two-class CNN with channel and spatial refinement branches and additive fusion. The training cell also uses augmentation for validation. Callback objects are defined but not passed into model.fit.


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten, Dense,
                                     BatchNormalization, GlobalAveragePooling2D,
                                     Input, Multiply, Add, Reshape, Activation, Dropout) # Import the Lambda layer
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import os
from tensorflow.keras.layers import Lambda
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

# Define a callable function for spatial attention
def spatial_avg_pool_fn(x):
    return tf.reduce_mean(x, axis=-1, keepdims=True)

def tri_attention(inputs):
    # Channel attention
    channel_avg_pool = GlobalAveragePooling2D()(inputs)
    channel_dense = Dense(inputs.shape[-1], activation='relu')(channel_avg_pool)
    channel_attention = Dense(inputs.shape[-1], activation='sigmoid')(channel_dense)
    channel_attention = Reshape((1, 1, inputs.shape[-1]))(channel_attention)
    channel_refined = Multiply()([inputs, channel_attention])

    # Spatial attention
    spatial_avg_pool = Lambda(spatial_avg_pool_fn)(inputs)  # Use function
    spatial_attention = Conv2D(1, (3, 3), padding='same', activation='sigmoid')(spatial_avg_pool)
    spatial_refined = Multiply()([inputs, spatial_attention])

    # Combined attention
    combined = Add()([channel_refined, spatial_refined])
    return combined


# Define the CNN Model with Tri-Attention Unit
def build_model(input_shape):
    inputs = Input(shape=input_shape)

    # First Convolution Block
    x = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2, 2))(x)

    # Second Convolution Block with Tri-Attention
    x = Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = tri_attention(x)
    x = MaxPooling2D((2, 2))(x)

    # Fully Connected Layers
    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)  # Increased the number of units
    x = Dropout(0.5)(x)  # Add Dropout to reduce overfitting
    x = Dense(2, activation='softmax')(x)

    model = Model(inputs, x)
    return model

# Parameters
input_shape = (128, 128, 3)
batch_size = 32
epochs = 50

# Data Augmentation and Loading (with Gaussian noise)
data_dir = '/content/drive/MyDrive/PlantVillage'  # Replace with your dataset path
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # 20% for validation
)

# Training Data with Gaussian Noise
train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(128, 128),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    classes=['Potato___Early_blight', 'Potato___Late_blight']
)

validation_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(128, 128),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    classes=['Potato___Early_blight', 'Potato___Late_blight']
)

# Build and Compile Model with Adam optimizer and lower learning rate
model = build_model(input_shape)
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Early stopping and learning rate reduction
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
# Train the Model
history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator
)


# Evaluate the Model
loss, accuracy = model.evaluate(validation_generator)
print(f"Validation Loss: {loss:.4f}, Validation Accuracy: {accuracy:.4f}")

# Plot training and validation accuracy/loss
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

## Historical evaluation on a filtered directory

This cell uses model_path, which is not defined anywhere in the supplied notebook. It requires an external model file and a separate filtered_dataset directory. See the review before interpreting its saved report.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

custom_objects = {
    'spatial_avg_pool_fn': spatial_avg_pool_fn,  # Add your function here
    'tri_attention': tri_attention              # Add tri_attention here as well
}

# Load the pre-trained model, providing the custom objects
model = tf.keras.models.load_model(model_path, custom_objects=custom_objects)

# Path to the filtered dataset
filtered_dir = '/content/drive/MyDrive/filtered_dataset'
if not os.path.exists(filtered_dir):
    raise FileNotFoundError(f"The filtered dataset directory was not found: {filtered_dir}")

# Load the filtered dataset using ImageDataGenerator
data_gen = ImageDataGenerator(rescale=1.0/255.0, validation_split=0.2)

validation_generator = data_gen.flow_from_directory(
    filtered_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

# Evaluate the model on validation data for precision, recall, F1, and accuracy
val_labels = []
val_predictions = []

# Loop through validation data to get predictions
for i in range(len(validation_generator)):
    x_val, y_val = validation_generator[i]
    val_labels.extend(np.argmax(y_val, axis=1))  # Get true labels
    predictions = model.predict(x_val)          # Get predictions
    val_predictions.extend(np.argmax(predictions, axis=1))

# Convert lists to numpy arrays
val_labels = np.array(val_labels)
val_predictions = np.array(val_predictions)

# Classification report
target_names = list(validation_generator.class_indices.keys())
print("Classification Report:")
print(classification_report(val_labels, val_predictions, target_names=target_names))

# Confusion Matrix
conf_matrix = confusion_matrix(val_labels, val_predictions)
print("Confusion Matrix:")
print(conf_matrix)

## Attention-refined feature visualization: first variant

Loads an external .keras model and inspects its Multiply layer outputs. These are attention-refined feature tensors, not raw attention-weight maps. The next cell contains another visualization variant.


In [ ]:
from tensorflow.keras.models import Model, load_model
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf  # Import TensorFlow
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # Import ImageDataGenerator

# Define a callable function for spatial attention (same as in training)
def spatial_avg_pool_fn(x):
    return tf.reduce_mean(x, axis=-1, keepdims=True)

# Define the tri_attention function in this cell as well
def tri_attention(inputs):
    # Channel attention
    channel_avg_pool = tf.keras.layers.GlobalAveragePooling2D()(inputs)
    channel_dense = tf.keras.layers.Dense(inputs.shape[-1], activation='relu')(channel_avg_pool)
    channel_attention = tf.keras.layers.Dense(inputs.shape[-1], activation='sigmoid')(channel_dense)
    channel_attention = tf.keras.layers.Reshape((1, 1, inputs.shape[-1]))(channel_attention)
    channel_refined = tf.keras.layers.Multiply()([inputs, channel_attention])

    # Spatial attention
    spatial_avg_pool = tf.keras.layers.Lambda(spatial_avg_pool_fn)(inputs)  # Use function
    spatial_attention = tf.keras.layers.Conv2D(1, (3, 3), padding='same', activation='sigmoid')(spatial_avg_pool)
    spatial_refined = tf.keras.layers.Multiply()([inputs, spatial_attention])

    # Combined attention
    combined = tf.keras.layers.Add()([channel_refined, spatial_refined])
    return combined


# Load the pre-trained model, providing the custom objects
model = load_model('/content/drive/MyDrive/potato_disease_classifierattention.keras',
                   custom_objects={'spatial_avg_pool_fn': spatial_avg_pool_fn,
                                   'tri_attention': tri_attention})  # Include 'tri_attention' as well


def visualize_images(images, titles, rows, cols, figsize=(15, 15)):
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten()
    for i, (img, title) in enumerate(zip(images, titles)):
        # Check the shape of the image and process if necessary
        if img.shape[-1] != 3:  # If not a 3-channel image (RGB)
            img = np.mean(img, axis=-1)  # Average across channels to get a grayscale image

        axes[i].imshow(img.squeeze(), cmap='viridis')
        axes[i].set_title(title, fontsize=8)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

# Redefine train_generator and validation_generator here:
# Parameters
input_shape = (128, 128, 3)
batch_size = 32
epochs = 50

# Data Augmentation and Loading (with Gaussian noise)
data_dir = '/content/drive/MyDrive/PlantVillage'  # Replace with your dataset path
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # 20% for validation
)

# Training Data with Gaussian Noise
train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(128, 128),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    classes=['Potato___Early_blight', 'Potato___Late_blight']
)

validation_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(128, 128),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    classes=['Potato___Early_blight', 'Potato___Late_blight']
)

# Create a new model to extract intermediate outputs for channel and spatial attention
intermediate_layer_model = Model(
    inputs=model.input,
    outputs=[
        model.get_layer('multiply').output,  # Changed to 'multiply' for channel attention output
        model.get_layer('multiply_1').output   # Changed to 'multiply_1' for spatial attention output
    ]
)

# Get a batch of data from the generator
batch_images, batch_labels = next(train_generator)

# Pass the batch through the intermediate model
channel_attention_outputs, spatial_attention_outputs = intermediate_layer_model.predict(batch_images)

# Select the first two images from each class
class_indices = train_generator.class_indices
class_names = list(class_indices.keys())

images_to_visualize = []
titles = []

for class_name, class_index in class_indices.items():
    # Find indices of images belonging to the current class
    class_indices_in_batch = np.where(np.argmax(batch_labels, axis=1) == class_index)[0]
    selected_indices = class_indices_in_batch[:2]  # Select first 2 images

    for idx in selected_indices:
        images_to_visualize.append(batch_images[idx])  # Original image
        images_to_visualize.append(channel_attention_outputs[idx])  # Channel attention output
        images_to_visualize.append(spatial_attention_outputs[idx])  # Spatial attention output

        titles.extend([
            f"{class_name} - Original",
            f"{class_name} - Channel Attention",
            f"{class_name} - Spatial Attention"
        ])

# Visualize the selected images in 3 columns
visualize_images(images_to_visualize, titles, rows=len(class_indices) * 2, cols=3, figsize=(15, 15))

## Attention-refined feature visualization: normalized variant

This repeats model loading and data-generator construction, then normalizes and averages refined feature channels for display. Normalization lacks a constant-map guard.


In [ ]:
from tensorflow.keras.models import Model, load_model
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf  # Import TensorFlow
from tensorflow.keras.preprocessing.image import ImageDataGenerator  # Import ImageDataGenerator

# Define a callable function for spatial attention (same as in training)
def spatial_avg_pool_fn(x):
    return tf.reduce_mean(x, axis=-1, keepdims=True)

# Define the tri_attention function in this cell as well
def tri_attention(inputs):
    # Channel attention
    channel_avg_pool = tf.keras.layers.GlobalAveragePooling2D()(inputs)
    channel_dense = tf.keras.layers.Dense(inputs.shape[-1], activation='relu')(channel_avg_pool)
    channel_attention = tf.keras.layers.Dense(inputs.shape[-1], activation='sigmoid')(channel_dense)
    channel_attention = tf.keras.layers.Reshape((1, 1, inputs.shape[-1]))(channel_attention)
    channel_refined = tf.keras.layers.Multiply()([inputs, channel_attention])

    # Spatial attention
    spatial_avg_pool = tf.keras.layers.Lambda(spatial_avg_pool_fn)(inputs)  # Use function
    spatial_attention = tf.keras.layers.Conv2D(1, (3, 3), padding='same', activation='sigmoid')(spatial_avg_pool)
    spatial_refined = tf.keras.layers.Multiply()([inputs, spatial_attention])

    # Combined attention
    combined = tf.keras.layers.Add()([channel_refined, spatial_refined])
    return combined

# Load the pre-trained model, providing the custom objects
model = load_model('/content/drive/MyDrive/potato_disease_classifierattention.keras',
                   custom_objects={'spatial_avg_pool_fn': spatial_avg_pool_fn,
                                   'tri_attention': tri_attention})

def visualize_images(images, titles, rows, cols, figsize=(15, 15), cmap='viridis'):
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten()
    for i, (img, title) in enumerate(zip(images, titles)):
        # Check if the image has more than 2 dimensions (likely an attention map)
        if len(img.shape) > 2 and img.shape[-1] != 3:  # Likely an attention map with more than 1 channel
            img = np.mean(img, axis=-1)  # Reduce to 2D by averaging channels
        if len(img.shape) == 2:  # Grayscale image
            axes[i].imshow(img, cmap=cmap)
        else:  # RGB image
            axes[i].imshow(img)

        axes[i].set_title(title, fontsize=8)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

# Define a function to normalize and enhance attention maps
def enhance_attention_map(attention_map):
    attention_map = np.squeeze(attention_map)  # Remove single-dimensional entries if present
    # Normalize to [0, 1]
    attention_map = (attention_map - np.min(attention_map)) / (np.max(attention_map) - np.min(attention_map))
    return attention_map

# Redefine train_generator and validation_generator here:
# Parameters
input_shape = (128, 128, 3)
batch_size = 32
epochs = 50

# Data Augmentation and Loading
data_dir = '/content/drive/MyDrive/PlantVillage'  # Replace with your dataset path
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # 20% for validation
)

train_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(128, 128),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    classes=['Potato___Early_blight', 'Potato___Late_blight']
)

validation_generator = train_datagen.flow_from_directory(
    data_dir,
    target_size=(128, 128),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    classes=['Potato___Early_blight', 'Potato___Late_blight']
)

# Create a new model to extract intermediate outputs for channel and spatial attention
intermediate_layer_model = Model(
    inputs=model.input,
    outputs=[
        model.get_layer('multiply').output,       # Channel attention output
        model.get_layer('multiply_1').output     # Spatial attention output
    ]
)

# Get a batch of data from the generator
batch_images, batch_labels = next(train_generator)

# Pass the batch through the intermediate model
channel_attention_outputs, spatial_attention_outputs = intermediate_layer_model.predict(batch_images)

# Select the first two images from each class
class_indices = train_generator.class_indices
class_names = list(class_indices.keys())

images_to_visualize = []
titles = []

for class_name, class_index in class_indices.items():
    # Find indices of images belonging to the current class
    class_indices_in_batch = np.where(np.argmax(batch_labels, axis=1) == class_index)[0]
    selected_indices = class_indices_in_batch[:2]  # Select first 2 images

    for idx in selected_indices:
        original_image = batch_images[idx]
        channel_attention_map = enhance_attention_map(channel_attention_outputs[idx])
        spatial_attention_map = enhance_attention_map(spatial_attention_outputs[idx])

        # Ensure attention maps are 2D for visualization
        channel_attention_map = np.mean(channel_attention_map, axis=-1) if channel_attention_map.ndim > 2 else channel_attention_map
        spatial_attention_map = np.mean(spatial_attention_map, axis=-1) if spatial_attention_map.ndim > 2 else spatial_attention_map

        images_to_visualize.extend([original_image, channel_attention_map, spatial_attention_map])
        titles.extend([
            f"{class_name} - Original",
            f"{class_name} - Channel Attention",
            f"{class_name} - Spatial Attention"
        ])

# Visualize the selected images in 3 columns
visualize_images(images_to_visualize, titles, rows=len(class_indices) * 2, cols=3, figsize=(15, 15))


## Model save and summary

This cell saves the current model variable. Earlier cells overwrite that variable by loading an external model, so this save is not proof that cell 1 training weights were exported.


In [ ]:
# Save the Model
model.save('/content/drive/MyDrive/potato_disease_classifierattention.keras')
model.summary()

## Historical evaluation on a 300-image subset

Selects up to 150 sorted images per potato class from the same PlantVillage directory. There is no independent test-set manifest or exclusion from the training/validation files. Its saved report differs from the training-cell validation result.


In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array # Import necessary functions
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import seaborn as sns

# Import LabelEncoder
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import load_model
custom_objects = {'tri_attention': tri_attention, 'spatial_avg_pool_fn': spatial_avg_pool_fn}
model = load_model('/content/drive/MyDrive/potato_disease_classifierattention.keras', custom_objects=custom_objects)


# Directories for the dataset
dataset_dir = '/content/drive/MyDrive/PlantVillage'
early_blight_dir = os.path.join(dataset_dir, 'Potato___Early_blight')
late_blight_dir = os.path.join(dataset_dir, 'Potato___Late_blight')

# Prepare test dataset: limit to 150 images per category
test_images = []
test_labels = []
categories = ['Potato___Early_blight', 'Potato___Late_blight']
label_encoder = LabelEncoder()
label_encoder.fit(categories)

for category, category_dir in zip(categories, [early_blight_dir, late_blight_dir]):
    label = label_encoder.transform([category])[0]
    image_count = 0

    for img_name in sorted(os.listdir(category_dir)):
        if image_count >= 150:  # Limit to 150 images per category
            break
        img_path = os.path.join(category_dir, img_name)

        try:
            img = load_img(img_path, target_size=(128, 128))  # Resize to model input size
            img_array = img_to_array(img) / 255.0  # Normalize
            test_images.append(img_array)
            test_labels.append(label)
            image_count += 1
        except Exception as e:
            print(f"Error loading image {img_name}: {e}")

# Convert lists to numpy arrays
test_images = np.array(test_images)
test_labels = np.array(test_labels)

# Predictions
predictions = model.predict(test_images)
predicted_labels = np.argmax(predictions, axis=1)

# Get confidence scores
confidence_scores = np.max(predictions, axis=1) # Get the maximum probability for each prediction

# Confusion matrix
cm = confusion_matrix(test_labels, predicted_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=categories, yticklabels=categories)
plt.title('Confusion Matrix (300 Photos)')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

# Classification report
print("Classification Report:")
print(classification_report(test_labels, predicted_labels, target_names=categories))

# Show individual predictions
for i in range(len(test_images)):
    plt.imshow(test_images[i])
    plt.title(f"True: {categories[test_labels[i]]}, Pred: {categories[predicted_labels[i]]}\nConfidence: {confidence_scores[i]:.2f}")
    plt.axis('off')
    plt.show()